# Financial Performance Prediction

本 Notebook 汇总本项目的输入审计、EDA、交叉验证、模型训练结果、OOF 融合、会计后处理和 submission 校验。所有表格和图形均从 `results/` 与 `figures/` 读取，避免手工抄写实验数值。

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
RESULTS = ROOT / 'results'
TABLES = RESULTS / 'tables'
FIGURES = ROOT / 'figures'
DELIVERABLES = ROOT / 'deliverables'

def show_table(path, n=10):
    df = pd.read_csv(path)
    display(df.head(n))
    return df

def show_fig(name):
    path = FIGURES / name
    display(Image(filename=str(path)))
    return path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


## 1. 输入文件与环境审计

In [ ]:
manifest = json.loads((RESULTS / 'input_manifest.json').read_text(encoding='utf-8'))
environment = json.loads((RESULTS / 'environment_audit.json').read_text(encoding='utf-8'))
print('Python:', environment['python_version'])
print('Executable:', environment['python_executable'])
print('Missing packages:', environment['missing_packages'])
pd.DataFrame(list(manifest['raw_files'].values())).loc[:, ['name', 'size_bytes', 'sha256']]

## 2. Schema 与数据质量审计

In [ ]:
schema = show_table(TABLES / 'schema_summary.csv')
missing = show_table(TABLES / 'missing_rate_by_column.csv', n=8)
duplicates = show_table(TABLES / 'duplicate_summary.csv')
target_summary = show_table(TABLES / 'target_summary.csv')

## 3. EDA 图表

In [ ]:
for fig in [
    'fig01_sector_distribution.png',
    'fig02_industry_top20.png',
    'fig03_missing_top20.png',
    'fig04_missing_by_quarter.png',
    'fig05_target_distributions.png',
    'fig06_lag_correlation_heatmap.png',
    'fig07_accounting_identity_error.png',
    'fig08_target_correlation_heatmap.png',
]:
    show_fig(fig)

## 4. 交叉验证与模型训练结果

In [ ]:
all_scores = show_table(TABLES / 'all_model_scores.csv', n=20)
final_scores = show_table(TABLES / 'final_model_scores.csv')
show_fig('fig09_model_comparison.png')
show_fig('fig10_target_score_heatmap.png')

## 5. OOF 融合与会计后处理

In [ ]:
blend_scores = show_table(TABLES / 'blend_scores.csv')
accounting_scores = show_table(TABLES / 'accounting_postprocess_scores.csv')
show_fig('fig11_oof_scatter.png')
show_fig('fig12_residual_distribution.png')
show_fig('fig13_blend_weights.png')

## 6. Submission 校验

In [ ]:
submission = pd.read_csv(DELIVERABLES / 'submission.csv')
sample = pd.read_csv(ROOT / 'sample_submission.csv')
assert submission.shape == sample.shape
assert list(submission.columns) == list(sample.columns)
assert submission['Id'].equals(sample['Id'])
values = submission.drop(columns=['Id'])
assert not values.isna().any().any()
assert values.applymap(lambda x: x == float('inf') or x == float('-inf')).sum().sum() == 0
print('submission rows, cols:', submission.shape)
print('submission sha256:', sha256_file(DELIVERABLES / 'submission.csv'))
submission.head()